In [1]:
import math
import matplotlib.pyplot as plt
from matplotlib.widgets import TextBox, Button
import numpy as np

TEAL = '#0D9488'
AMBER = '#F59E0B'
PURPLE = '#8B5CF6'
BLUE = '#3B82F6'
RED = '#EF4444'
GRAY = '#94A3B8'

plt.rcParams.update({
    'figure.figsize': (11, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'font.family': 'DejaVu Sans',
})

N = m1g cos θ
Fmotriz = m2g − m1g sin θ

Se |Fmotriz| ≤ µsN : a = 0 (EQUILÍBRIO ESTÁTICO)

Senão: a =
Fmotriz − µkN sinal(Fmotriz)
m1 + m2
(MOVIMENTO)

In [5]:
def calcular_aceleracao(m1, m2, theta_graus, coeficiente_estatico, coeficiente_cinetico, aceleracao_g):
    """
    Deduz e calcula a aceleração do sistema a partir da Segunda Lei de Newton (ΣF = m*a).

    Parâmetros:
    - m1: Massa do bloco no plano inclinado (kg)
    - m2: Massa do bloco/peso suspenso (kg)
    - theta_graus: Ângulo do plano inclinado
    - coeficiente_estatico: Coeficiente de atrito estático
    - coeficiente_cinetico: Coeficiente de atrito cinético
    - aceleracao_g: Aceleração da gravidade (m/s²)
    """
    theta_rad = np.radians(theta_graus)

    # 1. Força Normal (perpendicular ao plano inclinado): N = m1 * g * cos(theta)
    N = m1 * aceleracao_g * np.cos(theta_rad)

    # 2. Força Motriz Líquida: F_motriz = m2*g - m1*g*sin(theta)
    F_motriz = (m2 * aceleracao_g) - (m1 * aceleracao_g * np.sin(theta_rad))

    # 3. Limite do Atrito Estático: |F_motriz| <= coeficiente_estatico * N
    atrito_estatico_max = coeficiente_estatico * N

    if np.abs(F_motriz) <= atrito_estatico_max:
        # Equilíbrio Estático: o atrito anula a força motriz
        a = 0.0
        status = "Equilíbrio Estático (a = 0)"
    else:
        # Movimento: considera o atrito cinético oposto ao sentido do movimento
        sinal_f = np.sign(F_motriz)
        atrito_cinetico = coeficiente_cinetico * N

        # ΣF = (m1 + m2) * a => a = (F_motriz - F_atrito_cinetico) / (m1 + m2)
        a = (F_motriz - (atrito_cinetico * sinal_f)) / (m1 + m2)
        status = "Em Movimento"

    return a, N, F_motriz, status

 $\omega = \frac{v}{R}$, $x(t) = R \cos(\omega t)$, $y(t) = R \sin(\omega t)$ e $F_c = m \omega^2 R$):

In [7]:
def avaliar_trajetoria_vetor(m1, m2, theta_graus, coeficiente_estatico, coeficiente_cinetico, aceleracao_g, R, t_max, num_pontos=200):
    """
    Avalia a cinemática e as relações circulares diretamente sobre um vetor de tempo
    usando np.linspace e as fórmulas de movimento e força centrípeta.
    """
    # Obtém a aceleração do sistema usando a função anterior
    a, N, F_motriz, status = calcular_aceleracao(
        m1, m2, theta_graus, coeficiente_estatico, coeficiente_cinetico, aceleracao_g
    )

    # Vetor de tempo com NumPy
    t = np.linspace(0, t_max, num_pontos)

    # Velocidade linear em função do tempo (v = v0 + a*t, assumindo v0 = 0)
    v = a * t

    # Velocidade angular: omega = v / R (protegendo contra divisão por zero)
    omega = np.where(R > 0, v / R, np.zeros_like(t))

    # Posições paramétricas: x(t) = R * cos(omega * t), y(t) = R * sin(omega * t)
    fase = omega * t
    x = R * np.cos(fase)
    y = R * np.sin(fase)

    # Força Centrípeta: Fc = m * omega^2 * R (onde m = m1 + m2)
    m_total = m1 + m2
    Fc = m_total * (omega**2) * R

    return {
        "tempo": t,
        "velocidade": v,
        "velocidade_angular": omega,
        "pos_x": x,
        "pos_y": y,
        "forca_centripeta": Fc,
        "aceleracao": a,
        "status": status
    }